# Run a review with `run_review`

The smallest way to drive the whole pipeline from Python: build a `ReviewRequest`, call
`agentic_review.agents.pipeline_agent.run_review`, and look at what comes back. This is the same call the
`agentic-review review` command makes.

- Select this project's `.venv` kernel (`uv sync` installs `ipykernel`).
- Put the provider key in `.env` as `AGENTIC_REVIEW_API_KEY` (`GROQ_API_KEY` is used as a fallback).
- Optional: `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_BASE_URL` in `.env` send every review below to
  Langfuse as a trace (section 1). Without them nothing is traced and nothing else changes.
- `REPO` must be a git repository with uncommitted changes. The default is the demo built by
  `scripts/make_review_demo.py`.

## 1 · Settings

In [33]:
import os
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# ---- change these -------------------------------------------------------------------------------------------------
REPO = PROJECT.parent / "review-demo"  # the repository to review
SCOPE = None                           # None = config/default, or "working-tree" | "range"
BASE = None                            # the base ref, only for SCOPE = "range" (section 5 runs one)
MODEL = "qwen/qwen3.8-27b"  
BASE_URL = "https://api.groq.com/openai/v1"  # NVIDIA NIM (SRS \u00a72.3, same default as AgentSettings.base_url)

# -------------------------------------------------------------------------------------------------------------------


def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, value = line.partition("=")
                values[key.strip()] = value.strip().strip('"')
    return values


ENV = {**os.environ, **read_env_file(PROJECT / ".env")}
if not ENV.get("AGENTIC_REVIEW_API_KEY") and ENV.get("GROQ_API_KEY"):
    ENV["AGENTIC_REVIEW_API_KEY"] = ENV["GROQ_API_KEY"]

print("repo:   ", REPO, "(exists)" if REPO.is_dir() else "(MISSING)")
print("api key:", "set" if ENV.get("AGENTIC_REVIEW_API_KEY") else "NOT SET")  # the value is never printed
print("langfuse:", "keys found" if ENV.get("LANGFUSE_PUBLIC_KEY") and ENV.get("LANGFUSE_SECRET_KEY") else "not configured")

repo:    d:\Projects\DEPI-industry-project\review-demo (exists)
api key: set


### Langfuse tracing (optional)

With the three `LANGFUSE_*` values in `.env`, every review in this notebook is sent to Langfuse as **one trace**.
Without them the next cell says so, and `traced_review` is exactly `run_review`.

| In the trace | What it is |
|---|---|
| Name `review-change-set` | The same for every run. What differs — the scope — is a tag; the model and the system prompt are metadata |
| Input and output | The request, and the verdict with its finding counts and degradations. Not the whole report |
| One span per pipeline node | `preflight`, `resolve_change_set`, … `build_report` |
| `review_agent` | The review reporting agent. Each model call is a `generation` with the model, token usage, the messages, and a link to the managed system prompt when one is in use |
| Session | Every run of this kernel, so they sit side by side in Langfuse's Sessions view |

- **What leaves your machine:** the diff under review and the model's replies go to your Langfuse project. The
  provider key and the Langfuse secret key are masked out of everything sent; pipeline state never holds them anyway
  (`tests/integration/test_trace_carries_no_credential.py`).
- **Run this cell before the first review.** Langfuse keeps one tracer per project key, and the first client created
  decides whether it records — and the pipeline's own prompt loader creates a non-recording one. `traced_review` says
  so if that happened; the fix is to restart the kernel.
- `run_review` takes no callbacks, so the Langfuse handler is attached through LangChain's configure hook, and only
  while `traced_review` is running.

In [ ]:
import time
from contextvars import ContextVar
from pathlib import Path

from langchain_core.tracers.context import register_configure_hook

from agentic_review import __version__
from agentic_review.agents.pipeline_agent import Reviewed, run_review
from agentic_review.core.assembly.prompt import load_instructions

TRACE_NAME = "review-change-set"  # one name for every run; what differs between runs goes in tags and metadata
SESSION_ID = f"quickstart-{time.strftime('%Y%m%d-%H%M%S')}"  # groups this kernel's runs in Langfuse's Sessions view
TRACES = []  # (label, trace URL) of every traced run, listed in the last section

langfuse = None
if ENV.get("LANGFUSE_PUBLIC_KEY") and ENV.get("LANGFUSE_SECRET_KEY"):
    from langfuse import Langfuse, propagate_attributes
    from langfuse.langchain import CallbackHandler
    from langfuse.span_filter import is_default_export_span

    SECRETS = [s for s in (ENV.get("AGENTIC_REVIEW_API_KEY"), ENV.get("LANGFUSE_SECRET_KEY")) if s]

    def mask_secrets(*, data, **_):
        """Applied to every input, output and metadata value before it leaves the machine."""
        if isinstance(data, str):
            for secret in SECRETS:
                data = data.replace(secret, "[REDACTED]")
            return data
        if isinstance(data, dict):
            return {key: mask_secrets(data=value) for key, value in data.items()}
        if isinstance(data, (list, tuple)):
            return [mask_secrets(data=value) for value in data]
        return data

    langfuse = Langfuse(
        public_key=ENV["LANGFUSE_PUBLIC_KEY"],
        secret_key=ENV["LANGFUSE_SECRET_KEY"],
        base_url=ENV.get("LANGFUSE_BASE_URL") or ENV.get("LANGFUSE_HOST"),
        environment=ENV.get("LANGFUSE_TRACING_ENVIRONMENT") or "development",
        release=__version__,
        mask=mask_secrets,
        # The graphs' routing functions are traced as spans of their own; the node that runs next already says which
        # way a route went, so they are left out to keep the trace readable.
        should_export_span=lambda span: is_default_export_span(span)
        and not span.name.lstrip("_").startswith("route_"),
    )
    # `run_review` takes no callbacks, so the handler is attached the way a globally configured tracer attaches itself:
    # LangChain consults this context variable for every run. It is set only while `traced_review` is running.
    _handler: ContextVar[CallbackHandler | None] = ContextVar("quickstart_langfuse_handler", default=None)
    register_configure_hook(_handler, inheritable=True)
    print("Langfuse:", "connected" if langfuse.auth_check() else "AUTH FAILED", "|", ENV.get("LANGFUSE_BASE_URL"))
    print("session: ", SESSION_ID)
else:
    print("Langfuse is not configured: reviews run exactly as before, and nothing is traced.")


def _request_view(request):
    """What the trace shows as its input: the request, and nothing from the environment."""
    view = {"cwd": Path(request.cwd).name, "scope": request.scope, "base": request.base, "head": request.head,
            "pr": request.pr, "ceiling": request.ceiling}
    return {key: value for key, value in view.items() if value is not None}


def _outcome_view(outcome):
    """What the trace shows as its output: the verdict a reader wants at a glance, not the whole report."""
    if not isinstance(outcome, Reviewed):
        return {"preflight": "failed", "message": outcome.message}
    report = outcome.report
    counts = {level: sum(f.severity == level for f in report.findings) for level in ("blocking", "important", "nit")}
    return {
        "verdict": report.verdict,
        "completionStatus": report.run.completion_status,
        "scope": report.change_summary.scope.description,
        "findings": counts,
        "degradations": [d.code for d in report.degradations],
    }


def _system_prompt():
    """The system prompt this run will use, and the Langfuse prompt version to link when it is a managed one."""
    source = load_instructions(os.environ).source  # the same call, on the same environment, that the pipeline makes
    if not source.startswith("langfuse:"):
        return source, None
    name, _, version = source.removeprefix("langfuse:").rpartition("@v")
    return source, {"name": name, "version": int(version)}


def traced_review(request, *, label=None, tags=(), **kwargs):
    """`run_review(request, env=ENV, **kwargs)` as one Langfuse trace. With Langfuse off, it is just `run_review`."""
    if langfuse is None:
        return run_review(request, env=ENV, **kwargs)
    scope = "pull-request" if request.pr else (request.scope or "configured-scope")
    prompt_source, prompt_link = _system_prompt()
    metadata = {
        "model": str(MODEL or "configured"),
        "provider_url": str(BASE_URL or "configured"),
        "scope": scope,
        "system_prompt": prompt_source,
    }
    with propagate_attributes(
        trace_name=TRACE_NAME,
        session_id=SESSION_ID,
        tags=["notebook", "quickstart", scope, *tags],
        metadata=metadata,
        prompt=prompt_link,  # links each generation to the managed prompt version, when one is in use
    ):
        with langfuse.start_as_current_observation(
            as_type="span", name=TRACE_NAME, input=_request_view(request)
        ) as root:
            token = _handler.set(CallbackHandler())
            try:
                outcome = run_review(request, env=ENV, **kwargs)
            finally:
                _handler.reset(token)
            failed = not isinstance(outcome, Reviewed)
            root.update(
                output=_outcome_view(outcome),
                level="WARNING" if failed else "DEFAULT",
                status_message=outcome.message.splitlines()[0] if failed else None,
            )
            trace_id = root.trace_id
    if not trace_id.strip("0"):
        # Langfuse keeps one tracer per project key, and the first client created decides whether it records. The
        # pipeline's own prompt loader creates a non-tracing one, so this cell has to run before the first review.
        print("NOT TRACED: a review ran in this kernel before Langfuse was set up. Restart the kernel and run from the top.")
        return outcome
    TRACES.append((label or scope, langfuse.get_trace_url(trace_id=trace_id)))
    print("trace:", TRACES[-1][1])
    return outcome

## 2 · Run the review

`port_factory` is optional. It is used here only so `MODEL` / `BASE_URL` above can override the repository's
configuration without editing any file; leave both `None` and `run_review` uses its default.

`traced_review(request, ...)` is `run_review(request, env=ENV, ...)` inside one Langfuse trace; with Langfuse off it
is the same call. It prints the trace's link.

In [34]:
import time

from agentic_review.agents.pipeline_agent import ReviewRequest, Reviewed, run_review
from agentic_review.agents.review_port import build_review_port


def override_model(agent_settings, credential):
    overrides = {k: v for k, v in {"model": MODEL, "base_url": BASE_URL}.items() if v}
    return build_review_port(agent_settings.model_copy(update=overrides), credential)



In [35]:

request = ReviewRequest(cwd=REPO, scope=SCOPE, base=BASE)

started = time.perf_counter()
result = traced_review(request, label="working tree", port_factory=override_model if (MODEL or BASE_URL) else None)
print(f"{type(result).__name__} in {time.perf_counter() - started:.1f}s")

Reviewed in 2.7s


## 3 · Read the result

`run_review` returns one of two things:

- `PreflightFailure` — the repository, configuration or credential was not usable; nothing was reviewed.
- `Reviewed` — a `ReviewReport`, even when something degraded (FR-11.2). Always read `degradations`.

In [36]:
from rich.console import Console

from agentic_review.core.output.terminal import render_terminal

if not isinstance(result, Reviewed):
    print("PREFLIGHT FAILED:\n" + result.message)
else:
    report = result.report
    render_terminal(report, Console(width=140, force_jupyter=False, force_terminal=False, no_color=True))
    if result.ignored_settings:
        print("ignored settings:", ", ".join(result.ignored_settings))

Scope: uncommitted changes in the working tree (default) · 7 files
Reviewed (4):
  src/format.js
  src/retry.ts
  src/search.ts
  src/users.ts
Listed, not reviewed (3):
  PLANTED_DEFECTS.md (not TypeScript or JavaScript)
  README.md (not TypeScript or JavaScript)
  src/legacy.ts (deleted)
Excluded (1):
  dist/bundle.js (default:build-output)

No checks were run in this build.
No related context was gathered in this build.

src/search.ts:4  blocking  security
  findUsersByName
  evidence: unverified: The diff shows the parameterized query was replaced with string concatenation of untrusted input, but I cannot verify the exact line number in the new file.

src/users.ts:6  important  error-handling
  fetchUser(id)
  evidence: unverified: The diff shows the await was removed, but I cannot verify the exact line number in the new file.

src/users.ts:14-18  blocking  correctness
  averageAge
  evidence: unverified: The diff shows the empty array guard was removed and the loop condition change

In [16]:
assert isinstance(result, Reviewed), "preflight failed - see the cell above"
report = result.report

print("verdict:   ", report.verdict)
print("completion:", report.run.completion_status)
print("model:     ", report.run.model, "@", report.run.provider)
print("tokens:    ", report.run.tokens_in, "in /", report.run.tokens_out, "out")
print("stages ms: ", report.run.stage_durations_ms.model_dump(by_alias=True, exclude_none=True))

print(f"\n{len(report.findings)} finding(s)")
for f in report.findings:
    print(f"- [{f.severity}/{f.category}] {f.file}:{f.start_line}-{f.end_line}  {f.rationale}")

print(f"\n{len(report.degradations)} degradation(s)" + ("  (a clean run)" if not report.degradations else ""))
for d in report.degradations:
    print(f"- {d.source} {d.code}: {d.detail}")

verdict:    blocking
completion: emitted
model:      openai/gpt-oss-120b @ api.groq.com
tokens:     4634 in / 2660 out
stages ms:  {'preflight': 63, 'changeSet': 265, 'assembly': 0, 'review': 7547, 'output': 0}

2 finding(s)
- [nit/maintainability] src/format.js:2-2  The variable `unused` is computed but never used, adding dead code that serves no purpose and may confuse readers.
- [blocking/error-handling] src/retry.ts:1-10  The catch block swallows errors and the function returns undefined after all attempts, making failures indistinguishable from a legitimate undefined result and losing error context.

1 degradation(s)
- CF-6 incomplete-file-coverage: No emit_finding call, accepted or rejected, ever named: src/search.ts, src/users.ts.


## 4 · The full report as JSON

The same document `agentic-review review --json` prints.

In [17]:
from agentic_review.core.output.json_out import to_json

print(to_json(report))

{
  "schemaVersion": "1.0",
  "run": {
    "startedAt": "2026-09-26T16:35:16Z",
    "durationMs": 7875,
    "stageDurationsMs": {
      "preflight": 63,
      "changeSet": 265,
      "assembly": 0,
      "review": 7547,
      "output": 0
    },
    "requestedDepth": null,
    "achievedDepth": null,
    "toolCallCeiling": 25,
    "investigationToolsEnabled": false,
    "toolCallsTotal": 2,
    "toolCallsEmission": 2,
    "toolCallsInvestigation": 0,
    "repairTurns": 1,
    "completionStatus": "emitted",
    "emitPath": "tool-call",
    "model": "openai/gpt-oss-120b",
    "provider": "api.groq.com",
    "sampling": {
      "temperature": 0.0,
      "seed": null
    },
    "tokensIn": 4634,
    "tokensOut": 2660,
    "budgetLimit": 20000,
    "budgetHit": false,
    "schemaVersion": "1.0",
    "toolVersion": "0.1.0"
  },
  "changeSummary": {
    "scope": {
      "kind": "working-tree",
      "baseRef": null,
      "headRef": null,
      "wasDefaulted": true,
      "description": "uncomm

## 5 · Review a branch range and a pull request

Two more scopes, both built on the same three-dot diff (`git diff base...head`):

| Scope | Request | Compares |
|---|---|---|
| `range` | `ReviewRequest(cwd=REPO, scope="range", base=..., head=...)` | What `head` changed since it left `base`. `head=None` means `HEAD`. The working tree is not read |
| `pull-request` | `ReviewRequest(cwd=anywhere, pr=URL, base=...)` | A public GitHub pull request against its base branch. `base=None` means the repository's default branch |

A pull request is read through a temporary `git clone --filter=blob:none --no-checkout`: history only, no files on
disk. Nothing from it is executed and no `.agentic-review.yml` is read from it (SRS FR-10.3d). The clone is deleted
when the run ends.

- **This leaves your machine:** git contacts github.com, and the pull request's diff is sent to the model provider.
- **The git-based reader is interim.** It is to be replaced by one backed by the official GitHub MCP server
  (ADR-0011). `run_review(..., pull_requests=...)` is where another reader plugs in.
- Everything below is `None` by default, so "Run All" skips the runs that need a branch or the network.

In [37]:
# ---- change these -------------------------------------------------------------------------------------------------
RANGE_BASE = None  # e.g. "main": the ref the branch left. None skips the range run
RANGE_HEAD = None  # e.g. "feature-x". None = HEAD
PR_URL = "https://github.com/microsoft/playwright/pull/43199"      # e.g. "https://github.com/OWNER/REPO/pull/123". None skips the pull-request run
PR_BASE = None     # the branch the pull request targets. None = the repository's default branch
# -------------------------------------------------------------------------------------------------------------------

port_factory = override_model if (MODEL or BASE_URL) else None


def show(outcome):
    """Render a run and state its scope. Returns the report, or None after a preflight failure."""
    if not isinstance(outcome, Reviewed):
        print("PREFLIGHT FAILED:\n" + outcome.message)
        return None
    render_terminal(outcome.report, Console(width=140, force_jupyter=False, force_terminal=False, no_color=True))
    scope = outcome.report.change_summary.scope
    print(f"\nscope: kind={scope.kind}  baseRef={scope.base_ref}  headRef={scope.head_ref}")
    print("       " + scope.description)
    return outcome.report

### A branch range

What `RANGE_HEAD` changed since it left `RANGE_BASE`, in `REPO`. The branch does not have to be checked out.

In [ ]:
if RANGE_BASE is None:
    print("RANGE_BASE is not set - skipped.")
else:
    range_request = ReviewRequest(cwd=REPO, scope="range", base=RANGE_BASE, head=RANGE_HEAD)
    outcome = traced_review(range_request, label="branch range", port_factory=port_factory)
    range_report = show(outcome)

RANGE_BASE is not set - skipped.


### A public pull request

`cwd` does not matter here: no local repository is used and no file is read from it. The last line confirms the
temporary clone is gone.

In [39]:
import tempfile

from agentic_review.core.changeset.pull_request import WORKSPACE_PREFIX


def temporary_clones():
    return {p.name for p in Path(tempfile.gettempdir()).glob(f"{WORKSPACE_PREFIX}*")}


if PR_URL is None:
    print("PR_URL is not set - skipped.")
else:
    before = temporary_clones()
    pr_request = ReviewRequest(cwd=Path.cwd(), pr=PR_URL, base=PR_BASE)
    started = time.perf_counter()
    pr_report = show(traced_review(pr_request, label="pull request", port_factory=port_factory))
    print(f"\n{time.perf_counter() - started:.1f}s, including the clone")
    print("temporary clones left behind:", sorted(temporary_clones() - before) or "none")

Scope: pull request microsoft/playwright#43199 against main, the default branch (b8147648b..af4a20e34) · 16 files
Reviewed (10):
  packages/playwright-core/src/coreBundle.ts
  packages/playwright-core/src/entry/mcp.ts
  packages/playwright-core/src/package.ts
  packages/playwright-core/src/server/bidi/bidiChromium.ts
  packages/playwright-core/src/server/bidi/bidiOverCdp.ts
  packages/playwright-core/src/server/registry/index.ts
  packages/playwright/src/package.ts
  tests/installation/playwright-core-bundle.spec.ts
  utils/build/build.js
  utils/check_deps.js
Listed, not reviewed (6):
  .github/workflows/publish_release.yml (not TypeScript or JavaScript)
  packages/playwright-core/src/DEPS.list (not TypeScript or JavaScript)
  packages/playwright-core/src/server/bidi/DEPS.list (not TypeScript or JavaScript)
  packages/playwright-core/src/server/registry/DEPS.list (not TypeScript or JavaScript)
  packages/playwright/src/DEPS.list (not TypeScript or JavaScript)
  utils/publish_all_packa

In [29]:
pr_report.run

RunMetadata(started_at='2026-10-09T15:10:30Z', duration_ms=63766, stage_durations_ms=StageDurations(preflight=11875, change_set=1407, assembly=0, review=50484, output=0), requested_depth=None, achieved_depth=None, tool_call_ceiling=25, investigation_tools_enabled=False, tool_calls_total=1, tool_calls_emission=1, tool_calls_investigation=0, repair_turns=1, completion_status='emitted', emit_path='tool-call', model='openai/gpt-oss-120b', provider='api.groq.com', sampling=Sampling(temperature=0.0, seed=None), tokens_in=12441, tokens_out=4888, budget_limit=20000, budget_hit=False, schema_version='1.0', tool_version='0.1.0')

### What a pull request that cannot be read looks like

Both are preflight failures: the run stops before any model call. The first needs no network. The second asks GitHub
for a repository that does not exist, so it runs only when `PR_URL` is set — and it must fail with a message, never
with a login prompt.

In [ ]:
# Traced too: a preflight failure is a short trace at level WARNING, tagged so it can be filtered out.
failure = {"tags": ["expected-failure"], "port_factory": port_factory}
malformed = traced_review(ReviewRequest(cwd=Path.cwd(), pr="acme/widgets#7"), label="malformed URL", **failure)
print(type(malformed).__name__)
print(malformed.message)

if PR_URL is not None:
    missing_url = "https://github.com/agentic-review-no-such-owner/no-such-repository/pull/1"
    missing = traced_review(ReviewRequest(cwd=Path.cwd(), pr=missing_url), label="missing repository", **failure)
    print("\n" + type(missing).__name__)
    print(missing.message if not isinstance(missing, Reviewed) else "unexpected: that pull request was read")

## 6 · Send and open the traces

Langfuse sends in the background. Flush before closing the kernel, so the last runs arrive.

In [ ]:
if langfuse is None:
    print("Langfuse is not configured: nothing was traced.")
else:
    langfuse.flush()
    print(f"{len(TRACES)} trace(s) sent in session {SESSION_ID}")
    for label, url in TRACES:
        print(f"- {label}: {url}")